In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

In [2]:
from src.pipeline import (
    validate_data,
    clean_sales_data,
    get_database_connection
)

In [10]:
def load_incremental_orders(clean_df, con):
    con.execute("""
        CREATE TABLE IF NOT EXISTS clean_orders AS
        SELECT *
        FROM clean_df
        WHERE 1 = 0""")
    con.execute("""
        INSERT INTO clean_orders
        SELECT *
        FROM clean_df AS n
        WHERE NOT EXISTS(
            SELECT 1
            FROM clean_orders AS c
            WHERE n.order_id = c.order_id)
        """)

In [3]:
import pandas as pd

In [5]:
new_orders_df = pd.DataFrame({"order_id":[1001,1002,1003],
                             "product":["Laptop","Monitor","Keyboard"],
                             "quantity":[1,2,1],
                             "price":[1200,300,100],
                             "order_date":["2026-01-01","2026-01-02","2026-01-03"]})

In [6]:
clean_new_orders_df = clean_sales_data(new_orders_df)

In [7]:
con = get_database_connection()

In [8]:
con.execute("""
    DROP TABLE IF EXISTS clean_orders""")

In [11]:
load_incremental_orders(clean_new_orders_df, con)

In [13]:
loaded_orders_df = con.execute("""
    SELECT *
    FROM clean_orders""").df()

In [14]:
display(loaded_orders_df)

,order_id,product,quantity,price,order_date,total_sales
0,1001,Laptop,1,1200,2026-01-01,1200
1,1002,Monitor,2,300,2026-01-02,600
2,1003,Keyboard,1,100,2026-01-03,100


In [15]:
load_incremental_orders(clean_new_orders_df, con)

In [18]:
con.execute("""
    SELECT COUNT(*) AS row_count
    FROM clean_orders""").df()

,row_count
0,3


In [19]:
second_batch_df = pd.DataFrame({"order_id":[1003,1004,1005],
                             "product":["Keyboard","Mouse","Laptop"],
                             "quantity":[1,2,1],
                             "price":[100,50,1200],
                             "order_date":["2026-01-03","2026-01-04","2026-01-05"]})

In [20]:
clean_second_batch_df = clean_sales_data(second_batch_df)

In [21]:
load_incremental_orders(clean_second_batch_df,con)

In [22]:
con.execute("""
    SELECT COUNT(*) AS row_count
    FROM clean_orders""").df()

,row_count
0,5


In [ ]:
con.execute("""
    SELECT
        order_id,
        product,
        total_sales
    FROM clean_orders
    ORDER BY order_id;
    """).df()